# Within-Group Diffusion Synthetic Acceleration

This tutorial adds within-group diffusion synthetic acceleration (WGDSA) to a scattering-dominated slab solve.

**Audience:** Users encountering slow within-group convergence.

**Prerequisites:** Inner transport solvers and groupset options.

## Compare unaccelerated and accelerated solves

Both runs use PETSc GMRES and the same material with a scattering ratio of 0.9. The second groupset enables `apply_wgdsa` and configures the diffusion correction. Vacuum boundaries keep this compact example representative of a finite transport problem. The console iteration histories show the convergence effect; the final fluxes verify that acceleration does not change the solution.

In [ ]:
from mpi4py import MPI
from pyopensn.aquad import GLProductQuadrature1DSlab
from pyopensn.context import Finalize
from pyopensn.mesh import OrthogonalMeshGenerator
from pyopensn.post import VolumePostprocessor
from pyopensn.solver import DiscreteOrdinatesProblem, SteadyStateSourceSolver
from pyopensn.source import VolumetricSource
from pyopensn.xs import MultiGroupXS

rank = MPI.COMM_WORLD.rank

def solve_with_wgdsa(enabled):
    mesh = OrthogonalMeshGenerator(node_sets=[[i / 20.0 for i in range(21)]]).Execute()
    mesh.SetUniformBlockID(0)
    xs = MultiGroupXS()
    xs.CreateSimpleOneGroup(sigma_t=1.0, c=0.9)
    source = VolumetricSource(block_ids=[0], group_strength=[1.0])
    quadrature = GLProductQuadrature1DSlab(n_polar=8, scattering_order=0)
    groupset = {
        "groups_from_to": (0, 0),
        "angular_quadrature": quadrature,
        "inner_linear_method": "petsc_gmres",
        "l_abs_tol": 1.0e-8,
        "l_max_its": 200,
        "gmres_restart_interval": 30,
    }
    if enabled:
        groupset.update(
            {
                "apply_wgdsa": True,
                "wgdsa_l_abs_tol": 1.0e-8,
                "wgdsa_l_max_its": 100,
                "wgdsa_solver_policy": "auto",
                "wgdsa_verbose": False,
            }
        )
    problem = DiscreteOrdinatesProblem(
        mesh=mesh,
        num_groups=1,
        groupsets=[groupset],
        xs_map=[{"block_ids": [0], "xs": xs}],
        volumetric_sources=[source],
    )
    solver = SteadyStateSourceSolver(problem=problem)
    solver.Initialize()
    solver.Execute()
    average = VolumePostprocessor(problem=problem, value_type="avg")
    average.Execute()
    return float(average.GetValue()[0][0])

unaccelerated_flux = solve_with_wgdsa(False)
accelerated_flux = solve_with_wgdsa(True)

## Verify the accelerated solution

WGDSA changes how the iterative system is solved, not the converged transport solution. The two volume-averaged scalar fluxes should therefore agree within the transport tolerance.

In [ ]:
flux_difference = abs(unaccelerated_flux - accelerated_flux)
if rank == 0:
    print(f"Unaccelerated average flux={unaccelerated_flux:.6e}")
    print(f"WGDSA average flux={accelerated_flux:.6e}")
    print(f"WGDSA flux difference={flux_difference:.6e}")
assert flux_difference < 1.0e-5
if "opensn_console" not in globals():
    from IPython import get_ipython
    if get_ipython() is not None:
        Finalize()
        MPI.Finalize()